### Imports
This cell imports essential data manipulation and visualization libraries like `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn`.

In [12]:
#Import the tools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### File Path Definition
This cell imports the `os` module for path operations and defines the file path for the primary customer churn Excel dataset (`Customer_Churn_Dataset.xlsx`).

In [13]:
import os

_path = '/content/sample_data/Customer_Churn_Dataset.xlsx'

### Data Loading
This cell loads various datasets from Excel files and a CSV. It loads `Customer_Churn_Analysis.xlsx` into `df_churn_analysis`, the main `Customer_Churn_Dataset.xlsx` into `df`, and specific sheets (`Customer_Demographics`, `Online_Activity`, `Customer_Service`, `Transaction_History`) into `demo`, `online`, `service`, and `trans` DataFrames, respectively. It also loads the `churn_watchlist.csv` into the `watchlist` DataFrame.

In [14]:
#to upload the file
df = pd.read_excel('/content/sample_data/Customer_Churn_Dataset.xlsx')
demo = pd.read_excel(_path, sheet_name="Customer_Demographics")
df = demo # Align df with demo as requested
display(df.head())
online = pd.read_excel(_path, sheet_name="Online_Activity")
service = pd.read_excel(_path, sheet_name="Customer_Service")
trans = pd.read_excel(_path, sheet_name="Transaction_History")


,CustomerID,Age,AgeGroup,Gender,MaritalStatus,Churn status,IncomeLevel
0,9,39,Adult,M,Divorced,NO,High
1,19,57,Adult,F,Divorced,NO,High
2,34,33,Adult,M,Widowed,NO,High
3,57,46,Adult,F,Divorced,NO,High
4,58,52,Adult,M,Single,NO,High


### Q1: Overall Churn Rate
This cell calculates and prints the overall customer churn rate from the main `df` (Customer Demographics) DataFrame.

In [15]:
# Q1 — Overall churn rate
overall_churn = (df["Churn status"] == "YES").mean() * 100
print(f"Q1. Overall churn rate: {overall_churn:.1f}% "
      f"({(df['Churn status']=='YES').sum()} of {len(df)} customers)\n")

Q1. Overall churn rate: 20.4% (204 of 1000 customers)



### Q2: Churn Rate by Login-Frequency-Risk Band
This cell imports `scipy.stats` for statistical analysis. It then generates a 'Frequency Risk' for each customer (Low, Medium, High) using a deterministic random seed, calculates and displays the churn rate based on these risk bands, and performs a Chi-square test to check for statistical significance between 'Frequency Risk' and 'Churn status'.

In [16]:
import scipy.stats as stats

# Q2 — Churn rate by login-frequency-risk

merged = online.merge(df[["CustomerID", "Churn status"]], on="CustomerID")

ct = pd.crosstab(merged["Frequency Risk"], merged["Churn status"])
chi2, p_chi2, dof, _ = stats.chi2_contingency(ct)

print("Q2. Churn rate by Frequency Risk band:")
churn_by_risk = merged.groupby("Frequency Risk")["Churn status"].apply(
    lambda s: (s == "YES").mean() * 100
).round(1)
print(churn_by_risk.to_string(), "\n")
print(f"    Chi-square test of independence: chi2={chi2:.2f}, p={p_chi2:.5f} "
      f"({'SIGNIFICANT' if p_chi2 < 0.05 else 'not significant'} at p < 0.05)\n")

Q2. Churn rate by Frequency Risk band:
Frequency Risk
HIGH      19.1
LOW       35.1
MEDIUM    17.7 

    Chi-square test of independence: chi2=14.37, p=0.00076 (SIGNIFICANT at p < 0.05)



### Q3: Retention Watchlist Export
This cell creates a 'Retention Watchlist' by filtering for customers with 'Low' frequency risk from the `online` DataFrame. It merges this with demographic information from the `demo` DataFrame and then exports this watchlist to a CSV file named `churn_watchlist.csv`.

In [17]:
import os
# Q3 — Retention watchlist: export the LOW-risk segment
watchlist = online[online["Frequency Risk"] == "LOW"].merge(
    demo[["CustomerID", "AgeGroup", "Gender", "MaritalStatus", "IncomeLevel"]],
    on="CustomerID", how="left"
)
_out_dir = os.path.dirname(_path)  # write next to wherever the source file was found
_out_path = os.path.join(_out_dir, "churn_watchlist.csv")
watchlist.to_csv(_out_path, index=False)
print(f"Q3. Watchlist exported: {len(watchlist)} customers -> {_out_path}\n")

Q3. Watchlist exported: 97 customers -> /content/sample_data/churn_watchlist.csv



### Q4: Ruled-Out Variables Analysis
This cell analyzes variables that are 'ruled out' as strong predictors of churn. It examines churn rates based on service resolution status by merging `service` and `demo` DataFrames, and compares average spending between churned and retained customers by merging `trans` and `demo` DataFrames.

In [18]:
# Q4 — Ruled-out variables (documented, not just skipped)

svc = service.merge(demo[["CustomerID", "Churn status"]], on="CustomerID")
resolution_churn = svc.groupby("ResolutionStatus")["Churn status"].apply(
    lambda s: (s == "YES").mean() * 100
).round(1)
print("Q4. Ruled out — churn rate by service resolution outcome (expect ~flat):")
print(resolution_churn.to_string())

spend = trans.merge(demo[["CustomerID", "Churn status"]], on="CustomerID")
spend_by_churn = spend.groupby("Churn status")["AmountSpent"].mean().round(2)
print("\n    Ruled out — average spend, churned vs retained (expect churned >= retained):")
print(spend_by_churn.to_string())

Q4. Ruled out — churn rate by service resolution outcome (expect ~flat):
ResolutionStatus
Resolved      20.5
Unresolved    20.7

    Ruled out — average spend, churned vs retained (expect churned >= retained):
Churn status
NO     250.05
YES    253.30


Recommendation: Focus retention efforts on customers in the 'LOW' frequency risk band, as they represent a significant segment with a higher propensity to churn. While this segment is valuable, understanding and mitigating the factors contributing to churn in this group could yield greater impact. Further investigation into specific behaviors or demographics within the 'LOW' risk segment is recommended to tailor targeted interventions.